In [39]:
import cirq

In [43]:
class PauliErrors:

    def __init__(self, qubits):
        self.qubits = list(qubits) #just making sure this is a list! not necessary for us but for sanity

        #making a dictionary to assign/keep track of which error is on which qubit
            #starting out with no errors, will be included in circuit later
        self.errors = {q: "I" for q in self.qubits}


    def error_injection(self, qubit, error):
        #actualy assigning the error to a qubit (editing dict)
        self.errors[qubit] = error
            #adds gate on qubit itslelf before circuit starts, not on the state


    #just the display fucntion: showing the current errors on each qubit
    def show_errors(self):
        print("current errors:")
        for q in self.qubits:
            print(f"{q}: {self.errors[q]}")



    def propogate_gate(self, gate):
        operation= gate.gate #type of gate (i.e. trasnformation being applied)
        qubits= gate.qubits #on which qubit

        #operations per specific gate:

        ##X gate:
        if isinstance(operation, cirq.XPowGate):
            q= qubits[0]
            pass

        ##Z gate:
        elif isinstance(operation, cirq.ZPowGate):
            q = qubits[0]
            pass

        ##H gate:
        elif isinstance(operation, cirq.HPowGate):
            q= qubits[0]


            if self.errors[q] == "X":
                self.errors[q] = "Z"

            elif self.errors[q] == "Z":
                self.errors[q] = "X"

            elif self.errors[q] == "Y":
                self.errors[q] = "Y"

        ##CNOT gate:
        elif isinstance(operation, cirq.CXPowGate):
            control =qubits[0]
            target= qubits[1]

            control_error = self.errors[control]
            target_error = self.errors[target]

            #X error on the control qubit spreads to the target qubit
            if control_error == "X":

                if target_error == "I":
                    self.errors[target]= "X"

                elif target_error == "X":
                    self.errors[target]= "I"


            #Z error on the target qubit goes backwards to the control qubit
            if target_error == "Z":

                if control_error == "I":
                    self.errors[control] = "Z"

                elif control_error == "Z":
                    self.errors[control] = "I"

        else:
            print("No CNOT gate", operation)


    def run(self, circuit):
        print("initial errors:")
        self.show_errors()

        step=1

        for moment in circuit: #for multiple gates simultaneously

            for gate in moment.operations:
                print("step: ", step, "\n")
                print("gate: ", gate)

                self.propogate_gate(gate) #using propogate_gate to update qubtis
                self.show_errors()
                step+=1

In [48]:
#testing it out on a circuit

q0, q1, q2 = cirq.LineQubit.range(3)

circuit = cirq.Circuit()

circuit.append(cirq.H(q1))
circuit.append(cirq.CNOT(q0, q1))
circuit.append(cirq.CNOT(q1, q2))
circuit.append(cirq.H(q2))

print(circuit)

0: ───────@───────────
          │
1: ───H───X───@───────
              │
2: ───────────X───H───


In [49]:
propogation = PauliErrors([q0, q1, q2])
propogation.error_injection(q0, "X")
print(propogation.errors)

propogation.run(circuit)

{cirq.LineQubit(0): 'X', cirq.LineQubit(1): 'I', cirq.LineQubit(2): 'I'}
initial errors:
current errors:
q(0): X
q(1): I
q(2): I
step:  1 

gate:  H(q(1))
current errors:
q(0): X
q(1): I
q(2): I
step:  2 

gate:  CNOT(q(0), q(1))
current errors:
q(0): X
q(1): X
q(2): I
step:  3 

gate:  CNOT(q(1), q(2))
current errors:
q(0): X
q(1): X
q(2): X
step:  4 

gate:  H(q(2))
current errors:
q(0): X
q(1): X
q(2): Z
